# Lakehouse Table Cleanup Utility

This notebook helps clean up tables in a Fabric lakehouse with support for:
- Schema-enabled and non-schema lakehouses
- Prefix-based filtering
- Selective table deletion (all or specific tables)
- Dry run mode to preview before deletion

**Use Cases:**
- Clean up old source system tables (e.g., `adp__*` tables)
- Remove all tables from a specific schema
- Delete specific tables from a schema
- Test cleanup operations safely with dry run

**⚠️ WARNING: Table deletion is permanent and cannot be undone!**

## 1. Configuration Parameters

In [ ]:
# Lakehouse configuration
LAKEHOUSE_NAME = "Silver_Lake"  # The lakehouse/database name

# Schema configuration (for schema-enabled lakehouses)
USE_SCHEMA = False                # Set to True if this is a schema-enabled lakehouse
SCHEMA_NAME = "adp"              # Schema name (only used if USE_SCHEMA = True)

# Filtering options
TABLE_PREFIX = "as__"            # Prefix to filter tables (leave empty "" for all tables)

# Deletion mode
DELETE_MODE = "prefix"           # Options: "all" (all tables in schema), "prefix" (tables matching prefix), "manual" (specific list)

# Manual table list (only used if DELETE_MODE = "manual")
MANUAL_TABLE_LIST = [            # List specific table names (without schema prefix)
    # "table1",
    # "table2",
    # "table3"
]

# Safety settings
DRY_RUN = True                   # Set to False to actually delete tables
REQUIRE_CONFIRMATION = True      # Require explicit confirmation before deletion

## 2. Import Libraries and Helper Functions

In [ ]:
from typing import List, Dict
import re

def get_table_list(
    lakehouse: str,
    schema: str = None,
    prefix: str = "",
    use_schema: bool = False
) -> List[Dict[str, str]]:
    """
    Get list of tables from lakehouse/schema.
    
    Args:
        lakehouse: Lakehouse/database name
        schema: Schema name (for schema-enabled lakehouses)
        prefix: Table prefix filter
        use_schema: Whether this is a schema-enabled lakehouse
    
    Returns:
        List of dictionaries with table information
    """
    print(f"\n📋 Discovering tables...")
    
    if use_schema and schema:
        # Schema-enabled lakehouse
        print(f"   Lakehouse: {lakehouse}")
        print(f"   Schema: {schema}")
        full_db_name = f"{lakehouse}.{schema}"
    else:
        # Non-schema lakehouse
        print(f"   Lakehouse: {lakehouse}")
        full_db_name = lakehouse
    
    try:
        # Get all tables from the lakehouse/schema
        all_tables = spark.catalog.listTables(full_db_name)
        
        tables = []
        for t in all_tables:
            table_info = {
                'name': t.name,
                'database': t.database,
                'tableType': t.tableType,
                'isTemporary': t.isTemporary
            }
            
            # Filter by prefix if specified
            if prefix and not t.name.startswith(prefix):
                continue
            
            # Skip temporary tables
            if t.isTemporary:
                continue
            
            tables.append(table_info)
        
        print(f"   ✓ Found {len(tables)} table(s)")
        if prefix:
            print(f"   Filter: Tables starting with '{prefix}'")
        
        return tables
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        raise


def format_table_name(
    table_name: str,
    lakehouse: str,
    schema: str = None,
    use_schema: bool = False
) -> str:
    """
    Format fully qualified table name for SQL operations.
    
    Args:
        table_name: Table name
        lakehouse: Lakehouse name
        schema: Schema name
        use_schema: Whether to use schema format
    
    Returns:
        Fully qualified table name with backticks
    """
    if use_schema and schema:
        return f"`{lakehouse}`.`{schema}`.`{table_name}`"
    else:
        return f"`{lakehouse}`.`{table_name}`"


def drop_tables(
    tables: List[Dict[str, str]],
    lakehouse: str,
    schema: str = None,
    use_schema: bool = False,
    dry_run: bool = True
) -> Dict:
    """
    Drop tables from lakehouse.
    
    Args:
        tables: List of table dictionaries
        lakehouse: Lakehouse name
        schema: Schema name
        use_schema: Whether to use schema format
        dry_run: If True, only preview without deleting
    
    Returns:
        Dictionary with results summary
    """
    results = {
        'dropped': [],
        'failed': []
    }
    
    print(f"\n{'='*80}")
    print(f"{'DRY RUN - ' if dry_run else ''}Dropping {len(tables)} table(s)")
    print(f"{'='*80}")
    if use_schema and schema:
        print(f"Location: {lakehouse}.{schema}")
    else:
        print(f"Location: {lakehouse}")
    print(f"{'='*80}\n")
    
    for idx, table in enumerate(tables, 1):
        table_name = table['name']
        full_name = format_table_name(table_name, lakehouse, schema, use_schema)
        
        print(f"[{idx}/{len(tables)}] {full_name}")
        
        if dry_run:
            print(f"   ✓ Would drop table (DRY RUN)")
            results['dropped'].append({
                'table': table_name,
                'full_name': full_name,
                'status': 'dry_run'
            })
            continue
        
        try:
            # Execute DROP TABLE
            spark.sql(f"DROP TABLE IF EXISTS {full_name}")
            print(f"   ✓ Dropped successfully")
            results['dropped'].append({
                'table': table_name,
                'full_name': full_name,
                'status': 'dropped'
            })
            
        except Exception as e:
            error_msg = str(e)
            print(f"   ❌ Failed: {error_msg}")
            results['failed'].append({
                'table': table_name,
                'full_name': full_name,
                'error': error_msg
            })
    
    return results


print("✓ Helper functions loaded")

## 3. Discover Tables

Find tables based on configuration

In [ ]:
# Validate configuration
if USE_SCHEMA and not SCHEMA_NAME:
    raise ValueError("SCHEMA_NAME must be specified when USE_SCHEMA is True")

if DELETE_MODE == "manual" and not MANUAL_TABLE_LIST:
    raise ValueError("MANUAL_TABLE_LIST must be specified when DELETE_MODE is 'manual'")

# Get tables based on mode
if DELETE_MODE == "manual":
    print(f"📋 Using manually specified table list ({len(MANUAL_TABLE_LIST)} tables)")
    # Create table info dictionaries for manual list
    tables_to_process = [
        {
            'name': table_name,
            'database': f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}" if USE_SCHEMA else LAKEHOUSE_NAME,
            'tableType': 'MANAGED',
            'isTemporary': False
        }
        for table_name in MANUAL_TABLE_LIST
    ]
else:
    # Auto-discover tables
    prefix_filter = TABLE_PREFIX if DELETE_MODE == "prefix" else ""
    
    tables_to_process = get_table_list(
        lakehouse=LAKEHOUSE_NAME,
        schema=SCHEMA_NAME if USE_SCHEMA else None,
        prefix=prefix_filter,
        use_schema=USE_SCHEMA
    )

# Validate we found tables
if not tables_to_process:
    print("\n⚠️ No tables found matching criteria. Nothing to delete.")
else:
    # Display preview
    print(f"\n📊 Tables to be deleted:")
    print("="*80)
    
    preview_count = min(20, len(tables_to_process))
    for table in tables_to_process[:preview_count]:
        full_name = format_table_name(
            table['name'],
            LAKEHOUSE_NAME,
            SCHEMA_NAME if USE_SCHEMA else None,
            USE_SCHEMA
        )
        print(f"  - {full_name}")
    
    if len(tables_to_process) > preview_count:
        print(f"  ... and {len(tables_to_process) - preview_count} more tables")
    
    print(f"\nTotal tables: {len(tables_to_process)}")

## 4. Confirmation Check

⚠️ **IMPORTANT**: Review the tables above before proceeding!

In [ ]:
# Skip this cell if no tables found
if not tables_to_process:
    print("No tables to process. Skipping confirmation.")
else:
    if REQUIRE_CONFIRMATION and not DRY_RUN:
        print("\n" + "="*80)
        print("⚠️  CONFIRMATION REQUIRED")
        print("="*80)
        print(f"\nYou are about to DELETE {len(tables_to_process)} table(s) from:")
        if USE_SCHEMA:
            print(f"  Location: {LAKEHOUSE_NAME}.{SCHEMA_NAME}")
        else:
            print(f"  Location: {LAKEHOUSE_NAME}")
        print(f"\n⚠️  THIS ACTION CANNOT BE UNDONE!")
        print(f"\nTo proceed with deletion:")
        print(f"  1. Review the table list above")
        print(f"  2. Set REQUIRE_CONFIRMATION = False in Cell 1")
        print(f"  3. Re-run this notebook from Cell 1")
        print(f"\nOr run in DRY_RUN mode first by setting DRY_RUN = True")
        print("="*80)
        
        raise Exception("Confirmation required. Update configuration and re-run.")
    elif DRY_RUN:
        print("\n✓ DRY RUN mode enabled - no tables will be deleted")
    else:
        print("\n✓ Confirmation bypassed - proceeding with deletion")

## 5. Execute Table Deletion

In [ ]:
# Skip if no tables
if not tables_to_process:
    print("No tables to delete.")
else:
    # Execute deletion
    results = drop_tables(
        tables=tables_to_process,
        lakehouse=LAKEHOUSE_NAME,
        schema=SCHEMA_NAME if USE_SCHEMA else None,
        use_schema=USE_SCHEMA,
        dry_run=DRY_RUN
    )

## 6. Results Summary

In [ ]:
# Skip if no tables
if not tables_to_process:
    print("No results to display.")
else:
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    
    print(f"\n✓ Dropped: {len(results['dropped'])}")
    if results['dropped'] and len(results['dropped']) <= 10:
        for item in results['dropped']:
            print(f"   - {item['full_name']}")
    elif results['dropped']:
        for item in results['dropped'][:5]:
            print(f"   - {item['full_name']}")
        print(f"   ... and {len(results['dropped']) - 5} more")
    
    print(f"\n❌ Failed: {len(results['failed'])}")
    if results['failed']:
        for item in results['failed']:
            print(f"   - {item['full_name']}: {item['error']}")
    
    print(f"\n{'='*80}")
    print(f"Total Tables: {len(tables_to_process)}")
    success_rate = len(results['dropped']) / len(tables_to_process) * 100 if tables_to_process else 0
    print(f"Success Rate: {len(results['dropped'])}/{len(tables_to_process)} ({success_rate:.1f}%)")
    print("="*80)
    
    if DRY_RUN:
        print("\n⚠️ This was a DRY RUN - no tables were actually deleted")
        print("Set DRY_RUN = False in Cell 1 to execute for real")

## Usage Examples

### Example 1: Delete all tables with prefix in non-schema lakehouse
```python
LAKEHOUSE_NAME = "Silver_Lake"
USE_SCHEMA = False
TABLE_PREFIX = "as__"
DELETE_MODE = "prefix"
DRY_RUN = True
```

### Example 2: Delete all tables in a schema
```python
LAKEHOUSE_NAME = "Silver_Lake"
USE_SCHEMA = True
SCHEMA_NAME = "adp"
TABLE_PREFIX = ""  # Empty = all tables
DELETE_MODE = "all"
DRY_RUN = True
```

### Example 3: Delete tables with prefix in a schema
```python
LAKEHOUSE_NAME = "Silver_Lake"
USE_SCHEMA = True
SCHEMA_NAME = "staging"
TABLE_PREFIX = "temp__"
DELETE_MODE = "prefix"
DRY_RUN = True
```

### Example 4: Delete specific tables from a schema
```python
LAKEHOUSE_NAME = "Silver_Lake"
USE_SCHEMA = True
SCHEMA_NAME = "adp"
DELETE_MODE = "manual"
MANUAL_TABLE_LIST = ["old_table1", "old_table2", "temp_table"]
DRY_RUN = True
```

## Notes

**Concept: Why clean up tables?**
- Remove old source system tables after migration (e.g., `adp__*` after creating `adp` schema)
- Clean up temporary or staging tables
- Remove entire schemas during refactoring
- Maintain lakehouse hygiene and reduce storage costs

**Safety Features:**
- **DRY_RUN**: Preview operations without executing
- **REQUIRE_CONFIRMATION**: Explicit confirmation step before deletion
- **Table listing**: Review all tables before deletion
- **Error handling**: Failed deletions are logged and don't stop the process

**Best Practices:**
1. Always run in DRY_RUN mode first
2. Review the table list carefully
3. Back up important data before deletion
4. Use prefix filtering to avoid deleting wrong tables
5. Test on a small subset first with manual mode